src/playground/basic_ranking/basic_ranking.py
Loads prices via yfinance, ranks assets using Ranking, prints latest table, saves CSV.

In [1]:
from __future__ import annotations
import pandas as pd
import yfinance as yf
from models import Ranking 
import warnings
warnings.filterwarnings("ignore")

In [2]:
TICKERS = ["AAPL", "MSFT", "GOOGL", "AMZN", "META", "NVDA"]
START = "2022-01-01"
END = "2024-12-31" 

In [3]:
def load_prices(tickers, start, end) -> pd.DataFrame:
    """
    Download adjusted prices; return DataFrame (dates x tickers).
    """
    print(f"[INFO] Downloading {tickers} from {start} to {end}")
    data = yf.download(
        tickers,
        start=start,
        end=end,
        auto_adjust=True,   
        progress=True,
        threads=False    
    )
    if data.empty:
        raise ValueError("No data returned. Check tickers or date range.")
    if isinstance(data.columns, pd.MultiIndex):
        prices = data["Close"].dropna(how="all")
    else:
        prices = data[["Close"]].rename(columns={"Close": tickers[0]})
    return prices.dropna()

In [4]:
def main():
    prices = load_prices(TICKERS, START, END)
    model = Ranking(lookback=60, min_obs=40, winsor_pct=0.01)
    print("[INFO] Computing scores and ranks...")
    scores, ranks = model.rank(prices, ascending=False)

    last_dt = ranks.index[-1]
    latest = pd.DataFrame({
        "score": scores.loc[last_dt].sort_values(ascending=False),
        "rank": ranks.loc[last_dt].sort_values()
    })
    print(f"\nRanking at {last_dt.date()}:\n")
    print(latest.to_string(float_format=lambda x: f"{x:,.4f}"))

    out = f"ranking_{last_dt.date()}.csv"
    latest.to_csv(out)
    print(f"\n[INFO] Saved latest ranking to {out}")

In [5]:
if __name__ == "__main__":
    main()

[INFO] Downloading ['AAPL', 'MSFT', 'GOOGL', 'AMZN', 'META', 'NVDA'] from 2022-01-01 to 2024-12-31


[                       0%                       ]

[****************      33%                       ]  2 of 6 completed

[**********************50%                       ]  3 of 6 completed

[**********************67%*******                ]  4 of 6 completed

[**********************83%***************        ]  5 of 6 completed

[*********************100%***********************]  6 of 6 completed

[*********************100%***********************]  6 of 6 completed

[INFO] Computing scores and ranks...



Ranking at 2024-12-30:

        score   rank
Ticker              
AMZN   0.1820 1.0000
AAPL   0.1796 2.0000
GOOGL  0.1360 3.0000
NVDA   0.0931 4.0000
MSFT   0.0336 5.0000
META   0.0239 6.0000

[INFO] Saved latest ranking to ranking_2024-12-30.csv
